In [1]:
import duckdb
import plotly.express as px
import os


In [6]:
#Percorso al file generato dal Mart
#parquet_path = 'out/data/mart/istat-delitti-denunciati/2024/mart_delitti.parquet'

In [1]:
import duckdb
import plotly.express as px
import os

# USA QUESTO PERCORSO (Verificato sui tuoi log precedenti)
parquet_path = '/Users/alessio/Documents/dataciviclab-workspace/dataset-incubator/out/data/mart/istat-delitti-denunciati/2024/mart_delitti.parquet'

# Verifica rapida prima di procedere
if os.path.exists(parquet_path):
    print("✅ FILE TROVATO! Procedo con l'analisi...")
    con = duckdb.connect()
    
    # 1. Grafico dei reati più comuni
    df_reati = con.execute(f"""
        SELECT codice_reato, SUM(numero_denunce) as totale
        FROM '{parquet_path}'
        WHERE anno = '2015'
        GROUP BY codice_reato
        ORDER BY totale DESC
        LIMIT 15
    """).df()

    fig1 = px.bar(df_reati, x='codice_reato', y='totale', 
                 title='Top 15 Reati - Anno 2015',
                 template='plotly_dark')
    fig1.show()

    # 2. Trend Incendi
    df_trend = con.execute(f"""
        SELECT anno, SUM(numero_denunce) as totale
        FROM '{parquet_path}'
        WHERE codice_reato = 'ARSON'
        GROUP BY anno
        ORDER BY anno
    """).df()

    fig2 = px.line(df_trend, x='anno', y='totale', 
                  title='Trend Incendi Dolosi (ARSON)',
                  markers=True)
    fig2.show()
else:
    print("❌ Il file non è lì. Esegui in una cella: !find /Users/alessio/Documents/dataciviclab-workspace/dataset-incubator -name mart_delitti.parquet")

✅ FILE TROVATO! Procedo con l'analisi...


In [2]:
# Analisi comparativa dei trend
df_all_trends = con.execute(f"""
    SELECT anno, codice_reato, SUM(numero_denunce) as totale
    FROM '{parquet_path}'
    WHERE codice_reato IN ('ARSON', 'THEFT', 'ROBBERY', 'MURDER') -- Aggiungi qui i codici che trovi
    GROUP BY anno, codice_reato
    ORDER BY anno
""").df()

fig_trend = px.line(df_all_trends, x='anno', y='totale', color='codice_reato',
                   title='Evoluzione Comparata dei Reati (2010-2015)',
                   labels={'totale': 'Numero Denunce', 'anno': 'Anno'},
                   markers=True)
fig_trend.show()

In [3]:
df_geo = con.execute(f"""
    SELECT codice_territorio, SUM(numero_denunce) as totale
    FROM '{parquet_path}'
    WHERE anno = '2015' AND codice_reato = 'ARSON'
    GROUP BY codice_territorio
    ORDER BY totale DESC
    LIMIT 20
""").df()

fig_geo = px.bar(df_geo, x='codice_territorio', y='totale', 
                title='Top 20 Territori per Incendi Dolosi (2015)',
                color='totale', color_continuous_scale='Reds')
fig_geo.show()

In [4]:
df_zero = con.execute(f"""
    SELECT 
        CASE WHEN numero_denunce = 0 THEN 'Zero Denunce' ELSE 'Almeno 1 Denuncia' END as status,
        COUNT(*) as conteggio
    FROM '{parquet_path}'
    GROUP BY 1
""").df()

fig_pie = px.pie(df_zero, values='conteggio', names='status', 
                title='Distribuzione delle segnalazioni (Presenza vs Assenza di reati)')
fig_pie.show()

In [5]:
lista_reati = con.execute(f"SELECT DISTINCT codice_reato FROM '{parquet_path}'").df()
print(lista_reati)

   codice_reato
0      MAFIAHOM
1      MOPETHEF
2     MOTORTHEF
3       ROADHOM
4      SHOPTHEF
5        SMUGGL
6     TRUCKTHEF
7      UNINTHOM
8         USURY
9      VEHITHEF
10    ATTEMPHOM
11      CARTHEF
12   CORRUPUN18
13      COUNTER
14      FOREARS
15      INTPROP
16      MANSHOM
17         RAPE
18       ROBBER
19      SWINCYB
20    TERRORHOM
21      CRIMASS
22     HOUSEROB
23    INFANTHOM
24     INTENHOM
25      MAFIASS
26       MENACE
27      OFFENCE
28       RECEIV
29      SHOPROB
30        THEFT
31        ARSON
32      ARTTHEF
33      BAGTHEF
34        BLOWS
35     CULPINJU
36       DAMAGE
37         DRUG
38       EXTORT
39      KIDNAPP
40     MASSMURD
41     MONEYLAU
42      OTHCRIM
43     PICKTHEF
44        PORNO
45      POSTROB
46       PROSTI
47     RAPEUN18
48      ROBBHOM
49    STREETROB
50          TOT
51       ATTACK
52      BANKROB
53     BURGTHEF
54    CYBERCRIM
55       DAMARS


In [6]:
# Creiamo il dizionario di mapping nel tuo Notebook
mapping_ita = {
    'INTENHOM': 'Omicidi volontari', 'THEFT': 'Furti', 'ROBBER': 'Rapine',
    'DRUG': 'Stupefacenti', 'ARSON': 'Incendi', 'CYBERCRIM': 'Crimini Informatici',
    'DAMAGE': 'Danneggiamenti', 'EXTORT': 'Estorsioni', 'RAPE': 'Violenza Sessuale',
    'MONEYLAU': 'Riciclaggio', 'USURY': 'Usura', 'KIDNAPP': 'Sequestro di Persona'
}

df_top_reati = con.execute(f"""
    SELECT codice_reato, SUM(numero_denunce) as totale
    FROM '{parquet_path}'
    WHERE anno = '2015' AND codice_reato != 'TOT'
    GROUP BY codice_reato
    ORDER BY totale DESC
    LIMIT 10
""").df()

# Applichiamo il nome in italiano
df_top_reati['reato_it'] = df_top_reati['codice_reato'].map(mapping_ita).fillna(df_top_reati['codice_reato'])

fig_pie = px.pie(df_top_reati, values='totale', names='reato_it', 
                 title='Composizione dei principali reati denunciati (2015)',
                 hole=0.4, color_discrete_sequence=px.colors.sequential.RdBu)
fig_pie.show()

In [ ]:
# 1. Estrazione dati con conversione anno in numero
df_gravi = con.execute(f"""
    SELECT 
        CAST(anno AS INTEGER) as anno, 
        codice_reato, 
        SUM(numero_denunce) as totale
    FROM '{parquet_path}'
    WHERE codice_reato IN ('MAFIASS', 'INTENHOM', 'MAFIAHOM', 'KIDNAPP')
    GROUP BY anno, codice_reato
    ORDER BY anno
""").df()

# 2. AGGIUNGIAMO IL MAPPING (mancava questo passaggio!)
mapping_ita = {
    'INTENHOM': 'Omicidi volontari', 
    'MAFIASS': 'Associazione Mafiosa', 
    'MAFIAHOM': 'Omicidi di Mafia', 
    'KIDNAPP': 'Sequestro di Persona'
}
df_gravi['reato_it'] = df_gravi['codice_reato'].map(mapping_ita).fillna(df_gravi['codice_reato'])

# 3. Creazione grafico
fig_line = px.line(df_gravi, x='anno', y='totale', color='reato_it',
                  title='Evoluzione Reati di Grave Allarme Sociale (2010-2015)',
                  markers=True)

# 4. Limitiamo l'asse X ai dati reali
fig_line.update_xaxes(range=[2010, 2015], dtick=1)
fig_line.show()

ValueError: Value of 'color' is not the name of a column in 'data_frame'. Expected one of ['anno', 'codice_reato', 'totale'] but received: reato_it